# Corporate AI Adoption - Exploratory Data Analysis

This notebook explores the Corporate AI Adoption dataset using the same reusable cleaning and analysis logic that powers the dashboard and report.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import RAW_DATA_PATH, PROCESSED_DATA_PATH, clean_data, detect_outliers_iqr
from src.analysis import (
    adoption_driver_summary,
    categorical_summary,
    dataset_overview,
    duplicate_summary,
    future_readiness_by_industry,
    grouped_summary,
    missing_value_summary,
    trend_by_year,
    univariate_numeric_summary,
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
pd.set_option('display.max_columns', 40)

import warnings
warnings.filterwarnings('ignore')

## 1. Load and Clean Data

In [ ]:
df = clean_data(RAW_DATA_PATH)
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_DATA_PATH, index=False)

print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]:,}')
df.head()

## 2. Dataset Overview and Quality Checks

In [ ]:
overview = dataset_overview(df)
overview

In [ ]:
missing_value_summary(df)

In [ ]:
duplicate_summary(df)

In [ ]:
df.dtypes.to_frame('dtype')

## 3. Outlier Detection

Outliers are identified with the IQR rule for review. They are not removed because very large AI investments and financial outcomes can be valid business records.

In [ ]:
detect_outliers_iqr(df)

## 4. Univariate Analysis

In [ ]:
univariate_numeric_summary(df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df['ai_adoption_level'], bins=40, kde=True, ax=axes[0])
axes[0].set_title('AI Adoption Level')
sns.histplot(df['ai_maturity_score'], bins=30, kde=True, ax=axes[1])
axes[1].set_title('AI Maturity Score')
sns.histplot(df['productivity_gain'], bins=40, kde=True, ax=axes[2])
axes[2].set_title('Productivity Gain')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.countplot(y='industry', data=df, order=df['industry'].value_counts().index, ax=axes[0])
axes[0].set_title('Records by Industry')
sns.countplot(y='country', data=df, order=df['country'].value_counts().index, ax=axes[1])
axes[1].set_title('Records by Country')
plt.tight_layout()
plt.show()

## 5. Bivariate Analysis

In [ ]:
industry_summary = grouped_summary(df, 'industry')
industry_summary[['industry', 'records', 'avg_adoption', 'avg_maturity', 'avg_productivity_gain', 'avg_roi_proxy']]

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=industry_summary, x='avg_adoption', y='industry')
plt.title('Average AI Adoption by Industry')
plt.xlabel('Average AI Adoption Level')
plt.ylabel('Industry')
plt.show()

In [ ]:
country_summary = grouped_summary(df, 'country')
country_summary[['country', 'records', 'avg_adoption', 'avg_maturity', 'advanced_share']].head(15)

In [ ]:
scale_summary = grouped_summary(df, 'company_scale_proxy')
scale_summary[['company_scale_proxy', 'records', 'avg_adoption', 'avg_maturity', 'avg_total_financial_impact', 'advanced_share']]

## 6. Multivariate Analysis

In [ ]:
trend = trend_by_year(df)
trend.head()

In [ ]:
plt.figure(figsize=(13, 6))
sns.lineplot(data=trend, x='year', y='avg_adoption', label='AI adoption')
sns.lineplot(data=trend, x='year', y='avg_productivity_gain', label='Productivity gain')
plt.axvline(2026, color='gray', linestyle='--', linewidth=1)
plt.title('AI Adoption and Productivity Trend')
plt.ylabel('Average rate')
plt.show()

In [ ]:
industry_region = df.pivot_table(
    index='industry',
    columns='region',
    values='ai_adoption_level',
    aggfunc='mean',
    observed=True,
)

plt.figure(figsize=(12, 7))
sns.heatmap(industry_region, annot=True, fmt='.1%', cmap='viridis')
plt.title('Average AI Adoption by Industry and Region')
plt.show()

## 7. Correlation and Driver Analysis

In [ ]:
numeric_cols = [
    'ai_adoption_level', 'ai_investment_usd', 'automation_rate',
    'cost_savings', 'revenue_impact', 'productivity_gain',
    'employee_ai_training_hours', 'ai_maturity_score', 'deployment_count',
    'total_financial_impact', 'roi_proxy'
]
corr = df[numeric_cols].corr()

plt.figure(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1)
plt.title('Correlation Matrix')
plt.show()

In [ ]:
adoption_driver_summary(df)

## 8. Business Question Summaries

In [ ]:
print('Top industry by adoption:')
display(grouped_summary(df, 'industry').head(3))

print('Top countries by adoption:')
display(grouped_summary(df, 'country').head(5))

print('Region comparison:')
display(grouped_summary(df, 'region'))

print('Company-scale proxy comparison:')
display(grouped_summary(df, 'company_scale_proxy'))

In [ ]:
future_readiness_by_industry(df)

## 9. Notes for Interpretation

- The dataset has no direct company-size column, so `company_scale_proxy` uses deployment-count quartiles.
- The dataset has no explicit barrier survey fields, so barriers are inferred from available metrics.
- Years after 2026 are future/projected records in this project.
- Correlation does not prove causation.